In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

"""
EDA for Weather Temperature Prediction Dataset
目标变量: contest-tmp2m-14d__tmp2m (未来14天2米气温)
10 张独立图: 目标分布 → 缺失值 → 月份箱线 → 周均序列 → NMME相关 → NMME散点 → 特征重要性 → 学习曲线 → PCA投影 → PCA方差
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from config import DATA_RAW, TARGET, SEASON_ORDER, SEASON_PALETTE, FIG_DIR
from src.preprocessing import load_and_clean, get_column_groups, build_feature_matrix
from src.feature_engineering import compute_ensemble_importance
from src.evaluate import feature_count_curve

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
})
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ─── 0. 加载 & 预处理 ────────────────────────────────────────
print("Loading data...")
df_raw = pd.read_csv(DATA_RAW)
miss_pct = df_raw.isnull().mean() * 100
miss_pct = miss_pct[miss_pct > 0].sort_values(ascending=False).head(20)

df = load_and_clean(DATA_RAW)
col_groups = get_column_groups(df)
nmme_tmp_cols = col_groups['nmme_tmp']
print(f"Shape: {df.shape}\n")
del df_raw


# ════════════════════════════════════════════════════════════════
# Fig 1 | 目标变量分布
# ════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df[TARGET].dropna(), bins=60, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(df[TARGET].mean(), color='crimson', linestyle='--', label=f'Mean={df[TARGET].mean():.1f}°C')
ax.axvline(df[TARGET].median(), color='orange', linestyle='--', label=f'Median={df[TARGET].median():.1f}°C')
ax.set_xlabel('Temperature (°C)'); ax.set_ylabel('Count')
ax.set_title('Target Variable Distribution (contest-tmp2m-14d)')
ax.legend()
plt.tight_layout()
fig.text(0.5, -0.02, '(a)', ha='center', fontsize=11)
plt.savefig(FIG_DIR / 'fig1_target_distribution.png', bbox_inches='tight')
plt.show()
print("[1/10] Saved fig1\n")


# ════════════════════════════════════════════════════════════════
# Fig 2 | 缺失值 Top 20
# ════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(miss_pct.index[::-1], miss_pct.values[::-1], color='salmon')
ax.set_xlabel('Missing %'); ax.set_title('Top 20 Columns by Missing %')
ax.axvline(5, color='gray', linestyle=':', linewidth=1)
plt.tight_layout()
fig.text(0.5, -0.02, '(b)', ha='center', fontsize=11)
plt.savefig(FIG_DIR / 'fig2_missing_values.png', bbox_inches='tight')
plt.show()
print("[2/10] Saved fig2\n")


# ════════════════════════════════════════════════════════════════
# Fig 3 | 月份温度箱线图
# ════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
data_by_month = [df[df['month'] == m][TARGET].dropna() for m in range(1, 13)]
bp = ax.boxplot(data_by_month, patch_artist=True, notch=True,
                medianprops=dict(color='black', linewidth=2))
colors = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, 12))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_xlabel('Month'); ax.set_ylabel('Temperature (°C)')
ax.set_title('Temperature Distribution by Month')
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
plt.tight_layout()
fig.text(0.5, -0.02, '(a)', ha='center', fontsize=11)
plt.savefig(FIG_DIR / 'fig3_monthly_boxplot.png', bbox_inches='tight')
plt.show()
print("[3/10] Saved fig3\n")


# ════════════════════════════════════════════════════════════════
# Fig 4 | 周均时间序列
# ════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
ts = df.groupby('startdate')[TARGET].mean().sort_index()
ts_weekly = ts.resample('W').mean()
ax.plot(ts_weekly.index, ts_weekly.values, color='steelblue', linewidth=1.2, alpha=0.8)
ax.fill_between(ts_weekly.index, ts_weekly.values, alpha=0.15, color='steelblue')
ax.set_xlabel('Date'); ax.set_ylabel('Mean Temperature (°C)')
ax.set_title('Weekly Mean Temperature Over Time')
plt.tight_layout()
fig.text(0.5, -0.02, '(b)', ha='center', fontsize=11)
plt.savefig(FIG_DIR / 'fig4_weekly_timeseries.png', bbox_inches='tight')
plt.show()
print("[4/10] Saved fig4\n")


# ════════════════════════════════════════════════════════════════
# Fig 5 | NMME 相关系数
# ════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 6))
corr_nmme = df[nmme_tmp_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
colors_bar = ['#d62728' if v < 0 else '#1f77b4' for v in corr_nmme.values]
ax.barh(corr_nmme.index, corr_nmme.values, color=colors_bar)
ax.set_xlabel("Pearson r with Target")
ax.set_title('NMME Temperature Models — Correlation with Target')
ax.axvline(0.9, color='orange', linestyle='--', linewidth=1, label='r = 0.9')
ax.legend()
plt.tight_layout()
fig.text(0.5, -0.02, '(a)', ha='center', fontsize=11)
plt.savefig(FIG_DIR / 'fig5_nmme_correlation.png', bbox_inches='tight')
plt.show()
print("[5/10] Saved fig5\n")


# ════════════════════════════════════════════════════════════════
# Fig 6 | NMME 散点图
# ════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 6))
nmme_mean_col = 'nmme-tmp2m-34w__nmmemean'
for season in SEASON_ORDER:
    sub = df[df['season'] == season].sample(
        min(500, len(df[df['season'] == season])), random_state=42)
    ax.scatter(sub[nmme_mean_col], sub[TARGET], alpha=0.4, s=8,
               color=SEASON_PALETTE[season], label=season)
lims = [min(df[nmme_mean_col].min(), df[TARGET].min()),
        max(df[nmme_mean_col].max(), df[TARGET].max())]
ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='y = x')
r = df[[nmme_mean_col, TARGET]].corr().iloc[0, 1]
ax.set_xlabel('NMME Ensemble Mean (°C)'); ax.set_ylabel('Observed Temp (°C)')
ax.set_title(f'NMME Mean vs Observed (r = {r:.3f})')
ax.legend(markerscale=2)
plt.tight_layout()
fig.text(0.5, -0.02, '(b)', ha='center', fontsize=11)
plt.savefig(FIG_DIR / 'fig6_nmme_scatter.png', bbox_inches='tight')
plt.show()
print("[6/10] Saved fig6\n")


# ════════════════════════════════════════════════════════════════
# Fig 7 | 特征重要性 (score >= 0.1)
# ════════════════════════════════════════════════════════════════
print("\nRunning feature selection models (this may take ~1-2 min)...")
X, y, feature_cols = build_feature_matrix(df, TARGET)
ensemble_score, full_score = compute_ensemble_importance(X, y)
n_selected = len(ensemble_score)

top_features = ensemble_score[ensemble_score >= 0.1].sort_values(ascending=False)
n_top = len(top_features)
print(f"Features with score >= 0.1: {n_top}")

fig, ax = plt.subplots(figsize=(10, max(6, n_top * 0.35)))
short_labels = [f.split('__')[-1][:35] for f in top_features.index]
ax.barh(short_labels, top_features.values, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Ensemble Score (avg of Lasso + ElasticNet + RF)')
ax.set_title(f'Features with Score ≥ 0.1 ({n_top} features)')
plt.tight_layout()
fig.text(0.5, -0.02, '(a)', ha='center', fontsize=11)
plt.savefig(FIG_DIR / 'fig7_feature_importance.png', bbox_inches='tight')
plt.show()
print("[7/10] Saved fig7\n")


# ════════════════════════════════════════════════════════════════
# Fig 8 | 特征数量 vs CV RMSE 学习曲线
# ════════════════════════════════════════════════════════════════
n_full = len(full_score)
custom_counts = sorted(set(
    [1, 2, 3, 5] +
    list(range(10, min(n_full, 50), 10)) +
    list(range(50, n_full + 1, 30)) +
    [n_selected, n_full]
))
feat_counts, cv_rmses = feature_count_curve(
    X, y, full_score, counts=custom_counts,
    cache_path=FIG_DIR.parent / 'cache' / 'learning_curve.csv')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(feat_counts, cv_rmses, 'o-', color='steelblue', linewidth=2, markersize=6)
ax.fill_between(feat_counts, cv_rmses, min(cv_rmses), alpha=0.1, color='steelblue')
best_n = feat_counts[np.argmin(cv_rmses)]
idx_selected = feat_counts.index(n_selected)
ax.axvline(n_selected, color='crimson', linestyle='--',
           label=f'Selected = {n_selected} features (RMSE={cv_rmses[idx_selected]:.3f})')
ax.scatter([n_selected], [cv_rmses[idx_selected]], color='crimson', s=80, zorder=5)
ax.set_xlabel('Number of Features (ranked by ensemble score)')
ax.set_ylabel('CV RMSE (°C)')
ax.set_title(f'Learning Curve: Feature Count vs CV Error (total {n_full} features)')
ax.legend()
plt.tight_layout()
fig.text(0.5, -0.02, '(b)', ha='center', fontsize=11)
plt.savefig(FIG_DIR / 'fig8_learning_curve.png', bbox_inches='tight')
plt.show()
print("[8/10] Saved fig8\n")



# ════════════════════════════════════════════════════════════════
# EDA 总结
# ════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("EDA SUMMARY")
print("="*60)
print(f"Target mean: {df[TARGET].mean():.2f}°C  |  std: {df[TARGET].std():.2f}°C")
print(f"Date range: {df['startdate'].min().date()} to {df['startdate'].max().date()}")
print(f"\nTop 5 features by ensemble score:")
for feat, score in ensemble_score.head(5).items():
    print(f"  {feat[:50]:<50} score={score:.3f}")
print(f"\nOptimal feature count (elbow): ~{best_n}")
print(f"Missing value columns (before cleaning): {(df.isnull().mean() > 0).sum()}")
print("="*60)

Loading data...
